In [19]:
"""
Hourly Data Assimilation & Spatial Interpolation using Universal Kriging (Elevation Drift)
More physically informed...

Overview:
----------
This script builds hourly, gridded predictor fields (temperature, RH, PLP, and MRoS proxy)
on a 1-km DEM grid using Universal Kriging with elevation as an external drift term.
The workflow auto-fits a per-hour variogram and interpolates each variable from available
station, satellite (IMERG), and citizen-science (MRoS) observations.

Pipeline:
----------
1. CONFIG:
   - Defines variables, kriging model settings, and hourly time window.
   - Uses a spherical variogram with automatic parameter fitting.

2. UTILITIES:
   - Handles time indexing, CRS management, and DEM grid setup.

3. DATA:
   - Loads hourly station, IMERG, and MRoS parquet data.
   - Filters all observations to the DEM area of interest (AOI).

4. DEM Utilities:
   - Ensures each observation has an elevation from the DEM.

5. INTERPOLATION:
   - Performs per-hour Universal Kriging with elevation as an external drift.
   - Automatically fits variograms (PyKrige internal auto-fit).
   - Chunked prediction across the DEM grid to manage memory.
   - Local neighborhood (kriging_neighbors) limits the number of nearby stations used. THIS ACTUALLY ISN'T COMPATIBLE WITH UK, SO NO LOCAL NEIGHBORHOOD

6. HOURLY LOOP:
   - Iterates over each hour and variable to build a full xarray Dataset.

7. OUTPUTS:
   - Saves CF-compliant NetCDF file with gridded hourly predictors.
   - Generates optional static quicklook PNG maps with station overlays.

Key Parameters:
----------------
- min_points:     Minimum observations per variable required for kriging.
- kriging_neighbors:  Number of nearest observations used per grid-cell prediction.
- variogram_model: Type of spatial autocorrelation model ("spherical" default).
- variogram_strategy: "auto" lets PyKrige fit parameters for each hour dynamically.

Outputs:
--------
- CF-compliant NetCDF:  hourly_predictors_1km_kriging_*.nc
- Quicklook PNG maps:   /outputs/hourly_pipeline/maps/

Notes:
-------
This version replaces earlier lapse-rate detrending with a physically informed
Universal Kriging approach that treats elevation as a continuous external drift,
reducing over-smoothing while preserving local terrain-driven gradients.
"""


# ============================ IMPORTS ============================
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import rasterio as rio
import rioxarray
from rasterio.warp import transform_bounds, calculate_default_transform, reproject, Resampling
from rasterio.transform import xy as rio_xy, rowcol as rio_rowcol
import xarray as xr
from pyproj import CRS, Transformer
import matplotlib.pyplot as plt
from tqdm import tqdm

# Kriging
from pykrige.uk import UniversalKriging


In [6]:
BASE_DIR = Path().resolve().parent
print("BASE_DIR:", BASE_DIR)

CONFIG = {
    # Time windows
    "wy_start":  "2024-10-01T00:00:00Z",
    "wy_end":    "2025-05-31T23:59:59Z",
    "test_start": "2025-03-30T00:00:00Z",   # narrow test window first
    "test_end":   "2025-04-02T23:00:00Z",
    # "test_start": "2024-10-01T00:00:00Z",   # Entire window
    # "test_end":   "2025-05-31T23:59:59Z",

    # Paths
    "dem_path":  BASE_DIR / "DEM_1km_clipped.tif",   # ensure projected (meters)
    "out_dir":   BASE_DIR / "outputs/hourly_pipeline",

    # Projection fallback if DEM CRS is geographic
    "proj_fallback": "EPSG:26911",  # UTM 11N

    # Data inputs (hourly parquets produced upstream)
    "stations_parquet": BASE_DIR / "outputs/hourly_pipeline/hourly_data/stations_hourly.parquet",
    "imerg_parquet":    BASE_DIR / "outputs/hourly_pipeline/hourly_data/imerg_hourly.parquet",
    "mros_parquet":     BASE_DIR / "outputs/hourly_pipeline/hourly_data/mros_hourly.parquet",

    # Variables
    "variables": [
        ("temp_air",       "station"),
        ("temp_dew",       "station"),
        ("temp_wet",       "station"),
        ("rh",             "station"),
        ("mros_plp_proxy", "mros"),
        ("plp",            "imerg"),
    ],

    "min_points": {  # per-variable minimum points
        "temp_air": 4, "temp_dew": 4, "temp_wet": 4, "rh": 4,
        "mros_plp_proxy": 2, "plp": 1
    },

    # Kriging/variogram
    "variogram_model": "spherical",        # keep spherical as default
    "kriging_chunk_size": 1500,             # predict grid in chunks; how many grid points get kriged per iteration
    # "kriging_neighbors": 20,  # number of nearest points used in each local prediction (controls smoothing and speed)

    "variogram_strategy": "auto", 
}

OUT_DIR = Path(CONFIG["out_dir"]); OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_DIR: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype


In [7]:
# ============================ UTILITIES ============================
def to_utc(dt_series: pd.Series) -> pd.DatetimeIndex:
    """Force timestamps to UTC, making naive → UTC-naive assumed in UTC."""
    dt = pd.to_datetime(dt_series, errors="coerce", utc=True)
    # If dt_series had naive datetimes and pandas assumed local, .tz_convert('UTC') not needed.
    return dt

def hourly_index(start_iso: str, end_iso: str) -> pd.DatetimeIndex:
    return pd.date_range(start=pd.to_datetime(start_iso), end=pd.to_datetime(end_iso),
                         freq="H", tz="UTC")

def print_time(ts):
    return pd.to_datetime(ts).strftime("%Y-%m-%d %H:%MZ")

In [8]:
# --------------------- DEM Loading & Grid Setup ------------------------

def load_dem(path, target_crs):
    """
    Load DEM, reproject if needed, and return:
        dem (2D array), profile, CRS object
    Ensures DEM is ALWAYS in target_crs for consistency.
    """
    with rio.open(path) as src:
        src_crs = CRS.from_user_input(src.crs)
        tgt_crs = CRS.from_user_input(target_crs)

        # If CRS matches target CRS, no reprojection needed
        if src_crs == tgt_crs:
            dem = src.read(1).astype(np.float32)
            profile = src.profile
            profile["crs"] = tgt_crs.to_wkt()
            return dem, profile, tgt_crs

        # Otherwise reproject
        print(f"Reprojecting DEM from {src_crs} → {tgt_crs}")

        transform, width, height = calculate_default_transform(
            src_crs, tgt_crs, src.width, src.height, *src.bounds
        )

        profile = src.profile.copy()
        profile.update({
            "crs": tgt_crs.to_wkt(),
            "transform": transform,
            "width": width,
            "height": height,
        })

        dem = np.zeros((height, width), dtype=np.float32)

        reproject(
            rio.band(src, 1), dem,
            src_transform=src.transform, src_crs=src_crs,
            dst_transform=transform, dst_crs=tgt_crs,
            resampling=Resampling.bilinear,
        )

        return dem, profile, tgt_crs


# ---- Load DEM with consistent CRS ----

dem_data, dem_profile, proj_crs = load_dem(
    CONFIG["dem_path"],
    CONFIG["proj_fallback"]
)

H, W = dem_profile["height"], dem_profile["width"]
T = dem_profile["transform"]

# ---- Build coordinate centers using the affine transform ----

x_centers = T.c + (np.arange(W) + 0.5) * T.a
y_centers = T.f + (np.arange(H) + 0.5) * T.e   # T.e is usually negative for north-up

# Full grid coordinate pairs (H*W x 2) – CORRECT WAY
Xg, Yg = np.meshgrid(x_centers, y_centers)     # Xg, Yg are H x W
grid_xy = np.column_stack([Xg.ravel(), Yg.ravel()])  # (H*W, 2)

# ---- Flatten DEM and identify valid cells ----

grid_elev = dem_data.ravel().astype(np.float32)
valid_points = np.isfinite(grid_elev)

grid_xy_valid = grid_xy[valid_points]
grid_elev_valid = grid_elev[valid_points]

print(f"DEM CRS: {proj_crs}")
print(f"DEM size: {W} x {H}, pixel ~{abs(T.a):.1f} m")
print(f"Valid DEM cells: {len(grid_xy_valid)}")


# -------- Create AOI bounding polygon (in WGS84) --------

def load_dem_aoi(dem_profile):
    """Return AOI bounding box in WGS84 based on the DEM profile."""
    bounds = rio.coords.BoundingBox(*rio.transform.array_bounds(
        dem_profile["height"], dem_profile["width"], dem_profile["transform"]
    ))

    aoi_wgs84 = transform_bounds(
        CRS.from_wkt(dem_profile["crs"]),
        "EPSG:4326",
        bounds.left, bounds.bottom, bounds.right, bounds.top,
        densify_pts=21
    )

    return box(aoi_wgs84[0], aoi_wgs84[1], aoi_wgs84[2], aoi_wgs84[3])

aoi_poly = load_dem_aoi(dem_profile)


DEM CRS: EPSG:26911
DEM size: 275 x 504, pixel ~1000.0 m
Valid DEM cells: 138600


In [9]:
print("DEM stats:")
print("  NaNs:", np.isnan(dem_data).sum())
print("  +Inf:", np.isinf(dem_data).sum())
print("  -Inf:", np.isneginf(dem_data).sum())
print("  shape:", dem_data.shape)
print("  min/max:", np.nanmin(dem_data), np.nanmax(dem_data))

DEM stats:
  NaNs: 0
  +Inf: 0
  -Inf: 0
  shape: (504, 275)
  min/max: 0.0 4110.9526


In [10]:
# ============================ DATA LOADING ============================

# Load hourly parquets (already generated upstream)
st_hr   = pd.read_parquet(CONFIG["stations_parquet"])
imerg_hr = pd.read_parquet(CONFIG["imerg_parquet"])
mros_hr  = pd.read_parquet(CONFIG["mros_parquet"])

# Time to UTC and filter window
for df, time_col in [(st_hr, "hour_utc"), (imerg_hr, "hour_utc"), (mros_hr, "hour_utc")]:
    df[time_col] = pd.to_datetime(df[time_col], utc=True, errors="coerce").dt.floor("h")

HOURS = hourly_index(CONFIG["test_start"], CONFIG["test_end"])  # inclusive hourly range

# Filter to AOI bbox in lon/lat

def filter_points_to_aoi(df: pd.DataFrame, aoi_poly) -> pd.DataFrame:
    g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326")
    poly = gpd.GeoSeries([aoi_poly], crs="EPSG:4326").iloc[0]
    mask = g.intersects(poly)
    return df.loc[mask.values].drop(columns=["geometry"], errors="ignore")

st_hr   = filter_points_to_aoi(st_hr, aoi_poly)
imerg_hr= filter_points_to_aoi(imerg_hr, aoi_poly)
mros_hr    = filter_points_to_aoi(mros_hr, aoi_poly)

print(len(st_hr), len(imerg_hr), len(mros_hr))


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\2193629452.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  return pd.date_range(start=pd.to_datetime(start_iso), end=pd.to_datetime(end_iso),


557766 6966000 7896


In [11]:
# ============================= 4) DEM utils =============================

def add_dem_elev_if_missing(st_df: pd.DataFrame, profile, proj_crs) -> pd.DataFrame:
    """Fill missing station elevations by nearest-neighbor sampling of DEM."""
    if "elev" not in st_df.columns:
        st_df = st_df.copy(); st_df["elev"] = np.nan
    need = st_df["elev"].isna()
    if not need.any():
        return st_df

    tf = Transformer.from_crs("EPSG:4326", proj_crs, always_xy=True)
    xx, yy = tf.transform(st_df.loc[need, "lon"].values, st_df.loc[need, "lat"].values)
    rr, cc = rio_rowcol(profile["transform"], xx, yy, op=round)
    rr = np.clip(rr, 0, profile["height"] - 1)
    cc = np.clip(cc, 0, profile["width"]  - 1)
    st_df = st_df.copy(); st_df.loc[need, "elev"] = dem_data[rr, cc]
    return st_df

In [12]:
# ============================= 5) INTERPOLATION =============================

def _project_lonlat_to_xy(lon, lat, dst_crs):
    tf = Transformer.from_crs("EPSG:4326", dst_crs, always_xy=True)
    return tf.transform(lon, lat)


def _select_variogram_params(strategy: str):
    # For now, we just use auto-fit; this exists if you later want fixed params
    return None if strategy == "auto" else CONFIG.get("variogram_fixed_params", None)


def krige_with_dem_drift(hour_points: pd.DataFrame,
                         grid_xy: np.ndarray,
                         proj_crs,
                         value_col: str,
                         min_points: int,
                         dem_data: np.ndarray,
                         x_centers: np.ndarray,
                         y_centers: np.ndarray) -> np.ndarray:
    """
    Universal Kriging with DEM elevation as external Z drift.
    ---------------------------------------------------------
    - Uses 'external_Z' drift (recommended for physical terrain effects).
    - Drift at observations = DEM elevation at station coordinates.
    - Drift at predictions = DEM elevation grid (external_drift).
    - PyKrige automatically extracts DEM drift at prediction points.

    Inputs:
        hour_points : DataFrame with columns: lon, lat, value_col
        grid_xy     : N x 2 array of grid coords (unused directly except for shapes)
        proj_crs    : CRS of the DEM
        value_col   : variable to interpolate
        min_points  : minimum required obs for UK
        dem_data    : 2D DEM array (H x W)
        x_centers   : vector of DEM x-coordinates (W,)
        y_centers   : vector of DEM y-coordinates (H,)
    """

    # 1. Filter valid points
    pts = hour_points.dropna(subset=[value_col, "lon", "lat"]).copy()
    if pts.empty or pts[value_col].notna().sum() < min_points:
        print(f"Not enough points for {value_col}")
        return np.full(grid_xy.shape[0], np.nan, dtype=np.float32)

    # 2. Project station lon/lat → DEM CRS
    px, py = _project_lonlat_to_xy(
        pts["lon"].values, pts["lat"].values, proj_crs
    )

    # 3. Sample DEM elevation at those projected station points
    rr, cc = rio_rowcol(
        dem_profile["transform"],
        px, py,  # station projected coords
        op=round
    )
    rr = np.clip(rr, 0, dem_data.shape[0] - 1)
    cc = np.clip(cc, 0, dem_data.shape[1] - 1)
    obs_elev = dem_data[rr, cc].astype(float)   # drift at observations (Z)

    vals = pts[value_col].values.astype(float)

    print(f"Fitting UK for {value_col} with {len(vals)} obs...")

    # 4. Build UK model with DEM-based external drift
    try:
        UK = UniversalKriging(
            px, py, vals,
            variogram_model=CONFIG["variogram_model"],
            variogram_parameters=None if CONFIG["variogram_strategy"]=="auto" else CONFIG["variogram_fixed"],
            drift_terms=["external_Z"],         # DEM elevation drift
            external_drift=dem_data,            # full DEM grid (H x W)
            external_drift_x=x_centers,         # x-coordinates W
            external_drift_y=y_centers          # y-coordinates H
        )
    except Exception as e:
        print(f"UK init failed for {value_col}: {e}")
        return np.full(grid_xy.shape[0], np.nan, dtype=np.float32)

    # 5. Predict entire grid (DEM drift extracted internally)
    try:
        grid_x = grid_xy[:, 0]
        grid_y = grid_xy[:, 1]

        z_pred, _ = UK.execute(
            "points", grid_x, grid_y,
            backend="vectorized"   # optional, fast
        )
        return np.asarray(z_pred, dtype=np.float32)

    except Exception as e:
        print(f"Prediction failed for {value_col}: {e}")
        return np.full(grid_xy.shape[0], np.nan, dtype=np.float32)


In [13]:
# ============================= 6) HOURLY LOOP =============================

coords = {"time": HOURS, "y": y_centers, "x": x_centers}
var_names = [v[0] for v in CONFIG["variables"]]
data_vars = {name: np.full((len(HOURS), H, W), np.nan, dtype=np.float32) for name in var_names}

for ti, t in enumerate(tqdm(HOURS, desc="Hourly surfaces", ncols=88)):
    st_t   = st_hr[st_hr["hour_utc"] == t]
    imerg_t= imerg_hr[imerg_hr["hour_utc"] == t]
    mros_t = mros_hr[mros_hr["hour_utc"] == t]

    # Ensure station elevs present for lapse
    st_t = add_dem_elev_if_missing(st_t, dem_profile, proj_crs)

    for name, src in CONFIG["variables"]:
        min_pts = CONFIG["min_points"].get(name, 3)

        if src == "station":
            if name not in st_t.columns:
                continue
            pts = st_t[["lon", "lat", "elev", name]].dropna(subset=[name])
        elif src == "imerg":
            pts = imerg_t.rename(columns={"plp": name})[["lon", "lat", name]].assign(elev=0.0)
        elif src == "mros":
            pts = mros_t.rename(columns={"mros_plp_proxy": name})[["lon", "lat", name]].assign(elev=0.0)
        else:
            continue

        if pts[name].notna().sum() < min_pts:
            print(f"    {name}: insufficient points ({pts[name].notna().sum()} < {min_pts})")
            continue
        
        # HANDLING FOR MRoS PROXY (discrete / constant cases): fractionalize and introduce small random noise to preserve ordinal meaning but allow nonzero semivariance
        if name == "mros_plp_proxy":
            pts[name] = pts[name]/100.0 + np.random.uniform(-0.02, 0.02, len(pts))
            print(f"MRoS proxy variance @ {t}: {pts[name].var():.4f}")

        try:

            vals = krige_with_dem_drift(
                hour_points=pts,
                grid_xy=grid_xy_valid,
                proj_crs=proj_crs,
                value_col=name,
                min_points=min_pts,
                dem_data=dem_data,
                x_centers=x_centers,
                y_centers=y_centers,
            )

            if len(pts):
                print(f"    min={pts[name].min():.3f}, max={pts[name].max():.3f}, "
                    f"mean={pts[name].mean():.3f}, var={pts[name].var():.6f}")
            if name == "mros_plp_proxy":
                vals = np.clip(vals * 100.0, 0.0, 100.0)

        except Exception as e:
            print(f"Kriging failed for {name} @ {t}: {e}")
            vals = np.full(grid_elev.shape, np.nan)
        
        vals_full = np.full(H * W, np.nan, dtype=np.float32)
        vals_full[valid_points] = vals
        data_vars[name][ti, :, :] = vals_full.reshape(H, W)


Hourly surfaces:   0%|                                           | 0/96 [00:00<?, ?it/s]

Fitting UK for temp_air with 58 obs...
    min=-1.278, max=19.444, mean=7.652, var=40.311422
Fitting UK for temp_dew with 58 obs...
    min=-21.111, max=6.667, mean=-6.902, var=43.377487
Fitting UK for temp_wet with 58 obs...
    min=-7.320, max=11.539, mean=1.631, var=25.062178
Fitting UK for rh with 58 obs...


Hourly surfaces:   1%|▎                                  | 1/96 [00:19<31:31, 19.91s/it]

    min=6.000, max=79.000, mean=34.182, var=202.863808
    mros_plp_proxy: insufficient points (0 < 2)
    plp: insufficient points (0 < 1)
Fitting UK for temp_air with 58 obs...
    min=-2.389, max=18.333, mean=6.432, var=38.884010
Fitting UK for temp_dew with 58 obs...
    min=-19.444, max=6.667, mean=-6.606, var=46.002888
Fitting UK for temp_wet with 58 obs...
    min=-7.384, max=10.833, mean=1.104, var=25.422796
Fitting UK for rh with 58 obs...


Hourly surfaces:   2%|▋                                  | 2/96 [00:36<28:32, 18.22s/it]

...

    min=0.000, max=100.000, mean=50.993, var=2055.237899
Fitting UK for temp_air with 57 obs...
    min=-8.111, max=17.222, mean=3.119, var=55.645600
Fitting UK for temp_dew with 57 obs...
    min=-13.520, max=5.000, mean=-5.725, var=28.505238
Fitting UK for temp_wet with 57 obs...
    min=-14.091, max=10.694, mean=-1.071, var=40.918346
Fitting UK for rh with 57 obs...
    min=30.000, max=95.000, mean=53.367, var=182.957900
MRoS proxy variance @ 2025-04-02 22:00:00+00:00: 0.1522
Fitting UK for mros_plp_proxy with 7 obs...
Prediction failed for mros_plp_proxy: singular matrix
    min=-0.016, max=0.989, mean=0.284, var=0.152175
Fitting UK for plp with 1350 obs...


Hourly surfaces:  99%|██████████████████████████████▋| 95/96 [1:38:11<01:45, 105.51s/it]

    min=0.000, max=100.000, mean=50.993, var=2055.237899
Fitting UK for temp_air with 58 obs...
    min=-7.611, max=17.222, mean=3.116, var=55.047031
Fitting UK for temp_dew with 58 obs...
    min=-12.437, max=4.444, mean=-5.807, var=25.393558
Fitting UK for temp_wet with 58 obs...
    min=-12.568, max=10.188, mean=-1.057, var=37.689964
Fitting UK for rh with 58 obs...
    min=27.000, max=95.000, mean=52.290, var=165.996568
MRoS proxy variance @ 2025-04-02 23:00:00+00:00: 0.0877
Fitting UK for mros_plp_proxy with 3 obs...
Prediction failed for mros_plp_proxy: singular matrix
    min=0.480, max=0.999, mean=0.657, var=0.087669
Fitting UK for plp with 1350 obs...


Hourly surfaces: 100%|████████████████████████████████| 96/96 [1:40:04<00:00, 62.55s/it]

    min=0.000, max=100.000, mean=50.993, var=2055.237899


In [21]:
# =========================== SAVE NETCDF ==============================

# ---- 1. Build Dataset ----
ds = xr.Dataset(
    {
        **{k: xr.DataArray(v, coords=coords, dims=("time","y","x"))
           for k, v in data_vars.items()},
        "elev": xr.DataArray(
            dem_data.astype(np.float32),
            coords={"y": y_centers, "x": x_centers},
            dims=("y","x"),
            attrs={"units": "m", "long_name": "DEM elevation"}
        ),
    },
    attrs={
        "title": "Hourly predictor stacks on 1-km grid (Universal Kriging w/ elevation drift)",
        "interpolation_method": "Universal Kriging (external drift = DEM elevation)",
        "variogram_model": CONFIG["variogram_model"],
        "variogram_strategy": CONFIG["variogram_strategy"],
        "test_window": f"{CONFIG['test_start']} → {CONFIG['test_end']}",
    }
)

# ---- 2. Assign CF spatial dimensions ----
ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)

# ---- 3. Write CRS using EPSG, not CRS object ----
crs_obj = CRS.from_user_input(dem_profile["crs"])
epsg_code = crs_obj.to_epsg()
if epsg_code is None:
    raise ValueError(f"Could not derive EPSG from CRS: {dem_profile['crs']}")
ds = ds.rio.write_crs(epsg_code, grid_mapping_name="spatial_ref")

# ---- 4. Write GeoTransform ----
ds = ds.rio.write_transform(dem_profile["transform"])

# force consistency
A = dem_profile["transform"]
ds.attrs["GeoTransform"] = f"{A.c} {A.a} {A.b} {A.f} {A.d} {A.e}"

# ensure grid_mapping attribute exists
for v in ds.data_vars:
    ds[v].attrs["grid_mapping"] = "spatial_ref"

# ---- 5. Ensure time has no timezone ----
if hasattr(ds.indexes.get("time", None), "tz") and ds.indexes["time"].tz is not None:
    ds = ds.assign_coords(time=ds.indexes["time"].tz_localize(None))

# ---- 6. Encoding & Chunking ----
def _chunks_for(da):
    if da.ndim == 3 and da.dims == ("time","y","x"):
        return (min(24, da.shape[0]), min(256, da.shape[1]), min(256, da.shape[2]))
    if da.ndim == 2 and da.dims == ("y","x"):
        return (min(256, da.shape[0]), min(256, da.shape[1]))
    return None

encoding = {}
for name, da in ds.data_vars.items():
    ch = _chunks_for(da)
    enc = {"zlib": True, "complevel": 4}
    if ch:
        enc["chunksizes"] = ch
    encoding[name] = enc

# ---- 7. Write NetCDF ----
out_nc = OUT_DIR / "hourly_predictors_1km_kriging_v3_test.nc"

ds.to_netcdf(out_nc, engine="netcdf4", encoding=encoding)

print(f"Wrote {out_nc} successfully.")
ds.close()


Wrote C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\hourly_predictors_1km_kriging_v3_test.nc successfully.


In [22]:
# -------------------- Quick Plotting ------------------------------------

from pyproj import CRS

def quicklook_hour(
    ds, t, st_t, mros_t, out_png,
    vars_to_show=("plp","mros_plp_proxy","temp_air","temp_dew","temp_wet","rh")
):
    # match time index
    times_ds = pd.to_datetime(ds.time.values).floor("h")
    t_floor  = pd.to_datetime(t).floor("h")
    if t_floor not in times_ds.values:
        print(f"No matching time {t_floor} in dataset for quicklook.")
        return
    ti = int(np.where(times_ds == t_floor)[0][0])

    # axes extent (xmin, xmax, ymin, ymax)
    xvals = ds["x"].values
    yvals = ds["y"].values
    xmin, xmax = float(np.min(xvals)), float(np.max(xvals))
    ymin, ymax = float(np.min(yvals)), float(np.max(yvals))
    extent = [xmin, xmax, ymin, ymax]

    keep = [v for v in vars_to_show if v in ds.data_vars]
    if not keep:
        print("No matching variables to plot.")
        return
    ncols, nrows = 3, int(np.ceil(len(keep)/3))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.8*nrows), squeeze=False)
    fig.suptitle(f"Quicklook @ {t_floor:%Y-%m-%d %H:%MZ}", fontsize=14)

    # dataset CRS (fallback to configured)
    target_crs = ds.rio.crs or CRS.from_user_input(CONFIG["proj_fallback"])
    tf = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)

    # --- project & CLIP stations ---
    st_x = np.empty(0)
    st_y = np.empty(0)
    if len(st_t):
        sx, sy = tf.transform(st_t["lon"].values, st_t["lat"].values)
        sx = np.asarray(sx); sy = np.asarray(sy)
        smask = (sx >= xmin) & (sx <= xmax) & (sy >= ymin) & (sy <= ymax) & np.isfinite(sx) & np.isfinite(sy)
        st_x, st_y = sx[smask], sy[smask]

    # --- project & CLIP MRoS ---
    mo_x = np.empty(0)
    mo_y = np.empty(0)
    if len(mros_t):
        mx, my = tf.transform(mros_t["lon"].values, mros_t["lat"].values)
        mx = np.asarray(mx); my = np.asarray(my)
        mmask = (mx >= xmin) & (mx <= xmax) & (my >= ymin) & (my <= ymax) & np.isfinite(mx) & np.isfinite(my)
        mo_x, mo_y = mx[mmask], my[mmask]

    for i, var in enumerate(keep):
        ax = axes[i // ncols, i % ncols]
        arr = ds[var].isel(time=ti).values

        # color scaling
        if var in ("plp", "mros_plp_proxy", "rh"):
            im = ax.imshow(arr, origin="upper", extent=extent, aspect="equal", vmin=0, vmax=100)
        else:
            im = ax.imshow(arr, origin="upper", extent=extent, aspect="equal")

        ax.set_title(var)
        ax.set_xlabel("Easting (km)")
        ax.set_ylabel("Northing (km)")
        ax.ticklabel_format(style="plain")   # disable 1e6 scientific format
        ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
        ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])

        # overlay
        if st_x.size:
            ax.scatter(st_x, st_y, s=15, c="white", edgecolor="k",
                    marker="o", linewidths=0.5, label="Stations", zorder=3)
        if mo_x.size:
            ax.scatter(mo_x, mo_y, s=25, c="red", edgecolor="k",
                    marker="^", linewidths=0.6, label="MRoS", zorder=3)

        ax.legend(loc="upper right", frameon=True, fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    # turn off any leftover panels
    for j in range(len(keep), nrows*ncols):
        axes[j // ncols, j % ncols].axis("off")

    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    fig.savefig(out_png, dpi=200)
    plt.close(fig)
    print(
        f"Saved quicklook: {out_png} | plotted {st_x.size} stations, {mo_x.size} MRoS (clipped to DEM)"
    )


# -------------------- Loop --------------------

quick_dir = Path(CONFIG["out_dir"]) / "maps"
quick_dir.mkdir(parents=True, exist_ok=True)

# day = "2025-03-04"
# all_times = pd.to_datetime(ds.time.values).floor("h")  # dataset times
# mask = all_times.normalize() == pd.to_datetime(day)
# sample_hours = all_times[mask]
sample_hours = pd.to_datetime(ds.time.values)[::max(1, len(ds.time)//20)]
# print(f"Found {len(sample_hours)} timesteps on {day}")

for t in sample_hours:
    t_floor = pd.to_datetime(t).floor("h")  # tz-naive

    # Ensure obs times are made tz-naive before comparison
    st_t   = st_hr[st_hr["hour_utc"].dt.tz_convert(None).dt.floor("h") == t_floor]
    mros_t = mros_hr[mros_hr["hour_utc"].dt.tz_convert(None).dt.floor("h") == t_floor]

    print(f"[{t_floor}] Stations: {len(st_t)}, MRoS: {len(mros_t)}")

    quicklook_hour(ds, t_floor, st_t, mros_t,
                   out_png=quick_dir / f"test3_univ_kriging_quick_{print_time(t_floor).replace(':','-')}.png")


[2025-03-30 00:00:00] Stations: 58, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_univ_kriging_quick_2025-03-30 00-00Z.png | plotted 58 stations, 0 MRoS (clipped to DEM)
[2025-03-30 04:00:00] Stations: 57, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_univ_kriging_quick_2025-03-30 04-00Z.png | plotted 57 stations, 0 MRoS (clipped to DEM)
[2025-03-30 08:00:00] Stations: 58, MRoS: 0


...

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_univ_kriging_quick_2025-04-02 12-00Z.png | plotted 58 stations, 1 MRoS (clipped to DEM)
[2025-04-02 16:00:00] Stations: 58, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_univ_kriging_quick_2025-04-02 16-00Z.png | plotted 58 stations, 0 MRoS (clipped to DEM)
[2025-04-02 20:00:00] Stations: 58, MRoS: 5


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_22564\889464137.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_univ_kriging_quick_2025-04-02 20-00Z.png | plotted 58 stations, 5 MRoS (clipped to DEM)
